In [0]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load your feature data from the Gold layer
gold_df = spark.read.table("vp_catalog_f1_project.gold.pit_stop_features")
data_pd = gold_df.toPandas()

# 2. Define Features (X) and Label (y)
label = "pit_stop"
features = [
    "tire_age_laps",
    "lap_time_degradation",
    "lap_time_vs_race_avg",
    "lap_time_volatility",
    "lap_time"
]
X = data_pd[features]
y = data_pd[label]

# 3. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 4. Start an MLflow Experiment Run
with mlflow.start_run() as run:
    print("Starting MLflow Run:", run.info.run_uuid)
    mlflow.set_tag("Model", "Logistic Regression with Balanced Weights")

    # --- Train the Model with the FIX ---
    # The class_weight='balanced' parameter is the key change.
    lr = LogisticRegression(random_state=42, class_weight='balanced')
    lr.fit(X_train, y_train)

    # --- Evaluate the Model ---
    predictions = lr.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    # --- Log Metrics to MLflow ---
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # --- Log the Model to MLflow ---
    mlflow.sklearn.log_model(lr, "pit-stop-predictor")

    print("\nModel and metrics logged to MLflow successfully.")

Starting MLflow Run: 8939e55a760f411eb274347509865737
Accuracy: 0.9779
Precision: 0.4000
Recall: 1.0000
F1 Score: 0.5714


2025/08/31 11:23:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Model and metrics logged to MLflow successfully.
🏃 View run carefree-pug-745 at: https://adb-397370786462692.12.azuredatabricks.net/ml/experiments/2146099688511737/runs/8939e55a760f411eb274347509865737
🧪 View experiment at: https://adb-397370786462692.12.azuredatabricks.net/ml/experiments/2146099688511737
